In [ ]:
# !pip install -r ../requirements.txt

In [1]:
# import pkgs

import glob
import os
import torch
import pandas as pd
import numpy as np

from scaler.feature_scaler import GraphTargetScaler
import utils.pyg_dataset as pyg_dataset
from utils.utils import train_valid_test_split_indices

import warnings
warnings.filterwarnings('ignore')

# constant
EPW_FILE_PATH = "./EPW_Mar-Nov_SolarData_WithTemp_v2.csv"
SUN_NODE_ID = 0
PITCH_NODE_ID = 1
START_NODE_ID = 2
END_NODE_ID = 1850


/Users/maeng.k/anaconda3/envs/py313/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
folder_path = "./data"
node_csv_files = glob.glob(os.path.join(folder_path, "**", "*node*.csv"), recursive=True)
edge_csv_files = glob.glob(os.path.join(folder_path, "**", "*edge*.csv"), recursive=True)
node_csv_files.sort()
edge_csv_files.sort()

# File Sanity Check
for node_file in node_csv_files[:]:
    expected_edge_file = node_file.replace('node', 'edge')

    if expected_edge_file not in edge_csv_files:
        print(f"[Warning] Corresponding edge file not found for: {node_file}")
        print(f"          '{node_file}' will be removed from the node list.")
        node_csv_files.remove(node_file)

for edge_file in edge_csv_files[:]:
    expected_node_file = edge_file.replace('edge', 'node')
    if expected_node_file not in node_csv_files:
        print(f"[Warning] Corresponding node file not found for: {edge_file}")
        print(f"          '{edge_file}' will be removed from the edge list.")

        # edge_csv_files 리스트에서 해당 파일을 제거합니다.
        edge_csv_files.remove(edge_file)


print("\n--- Sanity Check Complete ---")
print("Updated node_csv_files:", len(node_csv_files))
print("Updated edge_csv_files:", len(edge_csv_files))



[Warning] Corresponding edge file not found for: ./data/0_close_0to90/0_close_0to90_Output_1374-42-86-0.65-10-25-18_case1374_node.csv
          './data/0_close_0to90/0_close_0to90_Output_1374-42-86-0.65-10-25-18_case1374_node.csv' will be removed from the node list.

--- Sanity Check Complete ---
Updated node_csv_files: 9378
Updated edge_csv_files: 9378


In [3]:

train_idx, valid_idx, test_idx = train_valid_test_split_indices(
    n_samples=len(node_csv_files),
    train_size=0.7,
    valid_size=0.15,
    test_size=0.15,
    random_state=42,
)

train_node_csv_files = [node_csv_files[i] for i in train_idx]
train_edge_csv_files = [edge_csv_files[i] for i in train_idx]

valid_node_csv_files = [node_csv_files[i] for i in valid_idx]
valid_edge_csv_files = [edge_csv_files[i] for i in valid_idx]

test_node_csv_files = [node_csv_files[i] for i in test_idx]
test_edge_csv_files = [edge_csv_files[i] for i in test_idx]

INFO:utils.utils:train_size: 0.7, valid_size: 0.15, test_size: 0.15
INFO:utils.utils:train_idx: 6564, valid_idx: 1407, test_idx: 1407


In [5]:
target_scaler = GraphTargetScaler(
    target_columns=['운동장_태양복사열'],
    feature_range=(0.0, 1.0)
).fit(train_node_csv_files)


# target_scaler = GraphTargetScaler(
#     target_columns=['운동장_태양복사열', '관중석 합산 태양복사열'],
#     feature_range=(0.0, 1.0)
# ).fit(train_node_csv_files)

target_scaler.save('target_scaler.pkl')


INFO:scaler.feature_scaler:Collecting target values from 6564 files...
INFO:scaler.feature_scaler:Progress: 100/6564
INFO:scaler.feature_scaler:Progress: 200/6564
INFO:scaler.feature_scaler:Progress: 300/6564
INFO:scaler.feature_scaler:Progress: 400/6564
INFO:scaler.feature_scaler:Progress: 500/6564
INFO:scaler.feature_scaler:Progress: 600/6564
INFO:scaler.feature_scaler:Progress: 700/6564
INFO:scaler.feature_scaler:Progress: 800/6564
INFO:scaler.feature_scaler:Progress: 900/6564
INFO:scaler.feature_scaler:Progress: 1000/6564
INFO:scaler.feature_scaler:Progress: 1100/6564
INFO:scaler.feature_scaler:Progress: 1200/6564
INFO:scaler.feature_scaler:Progress: 1300/6564
INFO:scaler.feature_scaler:Progress: 1400/6564
INFO:scaler.feature_scaler:Progress: 1500/6564
INFO:scaler.feature_scaler:Progress: 1600/6564
INFO:scaler.feature_scaler:Progress: 1700/6564
INFO:scaler.feature_scaler:Progress: 1800/6564
INFO:scaler.feature_scaler:Progress: 1900/6564
INFO:scaler.feature_scaler:Progress: 2000/656

In [7]:
target_scaler = GraphTargetScaler.load('target_scaler.pkl')

INFO:scaler.feature_scaler:Scaler loaded: target_scaler.pkl


In [8]:
train_datapyg_dataset, train_fail_file_paths = pyg_dataset.create_multi_graph_dataset(
    node_file_paths=train_node_csv_files,
    edge_file_paths=train_edge_csv_files,
    epw_file_path=EPW_FILE_PATH,
    target_columns=['운동장_태양복사열'],
    start_node_id=START_NODE_ID,
    end_node_id=END_NODE_ID,
    sun_node_id=SUN_NODE_ID,
    pitch_node_id=PITCH_NODE_ID,
    multi_task=False,
    use_node_coord=True,
    use_sun_coord=False,
    target_scaler=target_scaler,
)


valid_datapyg_dataset, valid_fail_file_paths = pyg_dataset.create_multi_graph_dataset(
    node_file_paths=valid_node_csv_files,
    edge_file_paths=valid_edge_csv_files,
    epw_file_path=EPW_FILE_PATH,
    target_columns=['운동장_태양복사열'],
    start_node_id=START_NODE_ID,
    end_node_id=END_NODE_ID,
    sun_node_id=SUN_NODE_ID,
    pitch_node_id=PITCH_NODE_ID,
    multi_task=False,
    use_node_coord=True,
    use_sun_coord=False,
    target_scaler=target_scaler,
)

test_datapyg_dataset, test_fail_file_paths = pyg_dataset.create_multi_graph_dataset(
    node_file_paths=test_node_csv_files,
    edge_file_paths=test_edge_csv_files,
    epw_file_path=EPW_FILE_PATH,
    target_columns=['운동장_태양복사열'],
    start_node_id=START_NODE_ID,
    end_node_id=END_NODE_ID,
    sun_node_id=SUN_NODE_ID,
    pitch_node_id=PITCH_NODE_ID,
    multi_task=False,
    use_node_coord=True,
    use_sun_coord=False,
    target_scaler=target_scaler,
)

print(f"Train fail file paths: {len(train_fail_file_paths)}")
print(f"Valid fail file paths: {len(valid_fail_file_paths)}")
print(f"Test fail file paths: {len(test_fail_file_paths)}")

print(len(train_datapyg_dataset))
print(len(valid_datapyg_dataset))
print(len(test_datapyg_dataset))


torch.save(train_datapyg_dataset, 'train_datapyg_dataset_scaled_y_sun_coord_false.pt' )
torch.save(valid_datapyg_dataset, 'valid_datapyg_dataset_scaled_y_sun_coord_false.pt' )
torch.save(test_datapyg_dataset, 'test_datapyg_dataset_scaled_y_sun_coord_false.pt' )

print("Saved train_datapyg_dataset.pt, valid_datapyg_dataset.pt, test_datapyg_dataset.pt")


  0%|          | 0/6564 [00:00<?, ?it/s]

  1%|          | 56/6564 [00:01<03:01, 35.89it/s]WARNING:utils.pandas_utils:Sun to pitchPt distance is 0 ---> Sun Set off ---> file will be passed
INFO:utils.pyg_dataset:Fail files: ./data/5_close_all_45/5_close_all_45_Output_1550-42-86-0.65-11-25-18_case1550_node.csv
  1%|▏         | 93/6564 [00:02<03:19, 32.42it/s]WARNING:utils.pandas_utils:Sun to pitchPt distance is 0 ---> Sun Set off ---> file will be passed
INFO:utils.pyg_dataset:Fail files: ./data/4_close_all_0/4_close_all_0_Output_1528-42-86-0.65-11-21-18_case1528_node.csv
  2%|▏         | 118/6564 [00:03<03:28, 30.88it/s]WARNING:utils.pandas_utils:Sun to pitchPt distance is 0 ---> Sun Set off ---> file will be passed
INFO:utils.pyg_dataset:Fail files: ./data/4_close_all_0/4_close_all_0_Output_1330-42-86-0.65-10-17-18_case1330_node.csv
  2%|▏         | 123/6564 [00:03<03:06, 34.58it/s]WARNING:utils.pandas_utils:Sun to pitchPt distance is 0 ---> Sun Set off ---> file will be passed
INFO:utils.pyg_dataset:Fail files: ./data/2_clos

Train fail file paths: 106
Valid fail file paths: 20
Test fail file paths: 23
6458
1387
1384
Saved train_datapyg_dataset.pt, valid_datapyg_dataset.pt, test_datapyg_dataset.pt


In [9]:
train_datapyg_dataset[0]


Data(x=[1849, 4], edge_index=[2, 5461], edge_attr=[5461, 2], y=[1, 1], scaled_y=[1, 1], global_x=[1, 4])

In [14]:
train_datapyg_dataset[0].edge_attr

tensor([[ 1.0000,  9.5018],
        [ 1.0000,  5.4670],
        [ 1.0000,  8.0366],
        ...,
        [ 1.0000, 27.8993],
        [ 1.0000, 25.5174],
        [ 1.0000, 23.1424]])

In [6]:
train_datapyg_dataset[0]


Data(x=[1849, 1], edge_index=[2, 5461], edge_attr=[5461, 2], y=[1, 1], global_x=[1, 4])

In [7]:
train_datapyg_dataset[0]


Data(x=[1849, 4], edge_index=[2, 5461], edge_attr=[5461, 2], y=[1, 1], global_x=[1, 4])